In [ ]:
# =========================
# MODEL 1 : PREPROCESSING
# =========================

import pandas as pd
import re
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/cnn_dailyMail News.csv")

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_article"] = df["article"].apply(clean_text)
df["clean_highlights"] = df["highlights"].apply(clean_text)

train, temp = train_test_split(df, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

print(train.shape, val.shape, test.shape)


(9435, 4) (1179, 4) (1180, 4)


In [ ]:

!pip install transformers datasets sentencepiece --quiet

In [ ]:
# ==============================
# MODEL 2 : LSTM Encoder-Decoder
# ==============================

!pip install tensorflow keras --quiet

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

# Use cleaned dataset from Model 1
train_df = train[["clean_article","clean_highlights"]].copy()
val_df   = val[["clean_article","clean_highlights"]].copy()

# Hyperparameters
num_words = 20000
max_enc_len = 200
max_dec_len = 40
embedding_dim = 128
lstm_units = 256

# -----------------------------
# PREPARE TOKENIZER
# -----------------------------
tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(
    list(train_df["clean_article"]) +
    list(train_df["clean_highlights"])
)

# Convert to sequences
def encode(texts, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    seq = pad_sequences(seq, maxlen=max_len, padding="post")
    return seq

encoder_input = encode(train_df["clean_article"], max_enc_len)

# Decoder inputs need <start> and <end> tokens
train_df["decoder_input"]  = train_df["clean_highlights"].apply(lambda x: "<start> " + x)
train_df["decoder_target"] = train_df["clean_highlights"].apply(lambda x: x + " <end>")

decoder_input  = encode(train_df["decoder_input"],  max_dec_len)
decoder_target = encode(train_df["decoder_target"], max_dec_len)

vocab_size = len(tokenizer.word_index) + 1

# -----------------------------
# BUILD ENCODER–DECODER MODEL
# -----------------------------

# Encoder
encoder_inputs = Input(shape=(max_enc_len,))
enc_embed = Embedding(vocab_size, embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(lstm_units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_embed)

# Decoder
decoder_inputs = Input(shape=(max_dec_len,))
dec_embed = Embedding(vocab_size, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(dec_embed, initial_state=[state_h, state_c])

# Final dense layer
decoder_dense = Dense(vocab_size, activation='softmax')
output_tokens = decoder_dense(decoder_outputs)

# Full model
model = Model([encoder_inputs, decoder_inputs], output_tokens)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()

# -----------------------------
# TRAIN MODEL
# -----------------------------
history = model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=32,
    epochs=1,
    validation_split=0.1
)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 200, 128)  │     19,968 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 40, 128)   │     19,968 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 256),     │    394,240 │ embedding_2[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 40, 256), │    394,240 │ embedding_3[0][0… │
│                     │ (None, 256),      │            │ lstm_2[0][1],     │
│                     │ (None, 256)]      │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 40, 156)   │     40,092 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 868,508 (3.31 MB)

 Trainable params: 868,508 (3.31 MB)

 Non-trainable params: 0 (0.00 B)

266/266 ━━━━━━━━━━━━━━━━━━━━ 270s 1s/step - loss: 0.8946 - val_loss: 0.0943


In [ ]:
!pip install evaluate rouge_score --quiet


In [ ]:
# =========================
# CORRECTED MODEL 3 : ROUGE SCORE
# =========================

!pip install evaluate rouge_score --quiet

import evaluate
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load ROUGE correctly
rouge = evaluate.load("rouge")

# Load trained T5 model
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small") # Corrected line
tokenizer = AutoTokenizer.from_pretrained("t5-small")

model.to("cpu") # Changed from "cuda" to "cpu"

preds = []
refs = []

# Evaluate on 50 samples
for i in range(50):
    article = test.iloc[i]["clean_article"]
    reference = test.iloc[i]["clean_highlights"]

    enc = tokenizer(
        "summarize: " + article,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cpu") # Changed from "cuda" to "cpu"

    output = model.generate(
        **enc,
        max_length=150,
        num_beams=4
    )

    pred = tokenizer.decode(output[0], skip_special_tokens=True)

    preds.append(pred)
    refs.append(reference)

# Compute ROUGE
rouge_scores = rouge.compute(predictions=preds, references=refs)

print("ROUGE Scores:")
print(rouge_scores)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

ROUGE Scores:
{'rouge1': np.float64(0.09638142829037968), 'rouge2': np.float64(0.024829807873286133), 'rougeL': np.float64(0.07415097611254898), 'rougeLsum': np.float64(0.07440962221921299)}


In [ ]:
# =========================
# MODEL 4 : BLEU SCORE
# =========================

!pip install sacrebleu nltk --quiet
import sacrebleu
import nltk
nltk.download("punkt")
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu

pred_list = preds
ref_list  = refs

# SacreBLEU
sacre = sacrebleu.corpus_bleu(pred_list, [ref_list])
print("SacreBLEU:", sacre.score)

# NLTK BLEU
scores = []
for p, r in zip(pred_list, ref_list):
    try:
        s = sentence_bleu([word_tokenize(r)], word_tokenize(p))
    except:
        s = 0
    scores.append(s)

print("Avg NLTK BLEU:", sum(scores)/len(scores))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.2 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


SacreBLEU: 0.5200433868057345
Avg NLTK BLEU: 0.0


In [ ]:
!pip install transformers datasets sentencepiece rouge_score evaluate --quiet


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train[["clean_article", "clean_highlights"]])
val_ds   = Dataset.from_pandas(val[["clean_article", "clean_highlights"]])


In [ ]:
from transformers import BertTokenizer, EncoderDecoderModel

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    "bert-base-uncased",  # Encoder
    "bert-base-uncased"   # Decoder
)

model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.encoder.vocab_size


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

In [ ]:
max_input_len = 256
max_output_len = 64

def preprocess(batch):
    # Encoder inputs
    inputs = tokenizer(
        batch["clean_article"],
        padding="max_length",
        truncation=True,
        max_length=max_input_len
    )

    # Decoder labels
    outputs = tokenizer(
        batch["clean_highlights"],
        padding="max_length",
        truncation=True,
        max_length=max_output_len
    )

    inputs["labels"] = outputs["input_ids"]
    return inputs

train_tok = train_ds.map(preprocess, batched=True)
val_tok   = val_ds.map(preprocess, batched=True)


Map:   0%|          | 0/9435 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer, model=model)


In [ ]:
# ================================================================
#   BERT2BERT Summarization (One Cell - 500 Steps, Old API Safe)
# ================================================================

!pip install --upgrade pip setuptools wheel --quiet
!pip install transformers datasets sentencepiece rouge_score evaluate --quiet

# Disable W&B
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

# Imports
from datasets import Dataset
from transformers import (
    BertTokenizer,
    EncoderDecoderModel,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate
import torch

# ================================================================
# Convert pandas → HF Dataset (from your preprocessing)
# ================================================================
train_ds = Dataset.from_pandas(train[["clean_article", "clean_highlights"]])
val_ds   = Dataset.from_pandas(val[["clean_article", "clean_highlights"]])

# ================================================================
# Load BERT2BERT
# ================================================================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# Explicitly set bos_token and eos_token for BertTokenizer to avoid generation config reset
tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token

model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    "bert-base-uncased",
    "bert-base-uncased"
)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.bos_token_id = tokenizer.cls_token_id # This will now be consistent with tokenizer.bos_token
model.config.eos_token_id = tokenizer.sep_token_id # This will now be consistent with tokenizer.eos_token
model.config.vocab_size = model.config.encoder.vocab_size # Added, was in previous setup

# ================================================================
# Preprocessing Function
# ================================================================
max_input_len = 256
max_output_len = 64

def preprocess(batch):
    enc = tokenizer(
        batch["clean_article"],
        padding="max_length",
        truncation=True,
        max_length=max_input_len
    )
    dec = tokenizer(
        batch["clean_highlights"],
        padding="max_length",
        truncation=True,
        max_length=max_output_len
    )
    enc["labels"] = dec["input_ids"]
    return enc

train_tok = train_ds.map(preprocess, batched=True)
val_tok   = val_ds.map(preprocess, batched=True)

collator = DataCollatorForSeq2Seq(tokenizer, model)

# ================================================================
# OLD-API SAFE TRAINING ARGUMENTS — ONLY 500 STEPS
# ================================================================
args = Seq2SeqTrainingArguments(
    output_dir="/content/bert2bert",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=3e-5,
    max_steps=50,        # 🔥 ONLY 500 STEPS!
    logging_steps=50,
    save_total_limit=2,
    report_to=[]          # disable wandb
)

# ================================================================
# Trainer (OLD API SAFE)
# ================================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=collator
)

trainer.train()

# ================================================================
# Inference Function
# ================================================================
def summarize(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    ids = model.generate(
        inputs["input_ids"],
        max_length=64,
        num_beams=4
    )
    return tokenizer.decode(ids[0], skip_special_tokens=True)

# Sample Prediction
print("\n================ SAMPLE SUMMARY ================")
sample = test.iloc[0]["clean_article"]
print("Generated Summary:\n", summarize(sample))

# ================================================================
# ROUGE SCORE (10 Samples)
# ================================================================
rouge = evaluate.load("rouge")
preds, refs = [], []

for i in range(10):
    art = test.iloc[i]["clean_article"]
    ref = test.iloc[i]["clean_highlights"]
    preds.append(summarize(art))
    refs.append(ref)

scores = rouge.compute(predictions=preds, references=refs)

print("\n================ ROUGE SCORES ================")
print(scores)


Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

Map:   0%|          | 0/9435 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

/tmp/ipython-input-4293625768.py:93: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 102, 'bos_token_id': 101}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:575: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe 

Step,Training Loss


Step,Training Loss
50,3.477000



================ SAMPLE SUMMARY ================
Generated Summary:
 action action in action to to to economic action to to to in to to to in to to to to to to to to results to improvements improvements to news to improvements improvements improvements results improvements in to improvements to to to news to to to to to improvements to improvements improvements improvements to improvements improvements to improvements improvements to to to

================ ROUGE SCORES ================
{'rouge1': np.float64(0.014620630861040068), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.011679454390451833), 'rougeLsum': np.float64(0.011679454390451833)}
